# xG Rolling Stats toevoegen aan trainingsdata
Voegt home_xg/away_xg toe inclusief rolling stats (alltime + last10, overall/home/away).

In [1]:
import pandas as pd
import numpy as np

# === LADEN ===
train = pd.read_csv('training_data.csv')
xg    = pd.read_csv('understat_xg.csv')

# Datum aanmaken
train['date'] = pd.to_datetime(train['timestamp']).dt.date.astype(str)
xg['date']    = xg['date'].astype(str)

print(f'Training data: {len(train)} rijen')
print(f'Understat xG:  {len(xg)} rijen')

FileNotFoundError: [Errno 2] No such file or directory: 'training_data.csv'

In [ ]:
# === TEAMNAMEN HARMONISEREN ===
NAME_MAP = {
    'Brighton':               'Brighton & Hove Albion',
    'Ipswich':                'Ipswich Town',
    'Leeds':                  'Leeds United',
    'Leicester':              'Leicester City',
    'Luton':                  'Luton Town',
    'Norwich':                'Norwich City',
    'Tottenham':              'Tottenham Hotspur',
    'West Ham':               'West Ham United',
    'Wolverhampton Wanderers':'Wolverhampton',
}

xg['home_team'] = xg['home_team'].replace(NAME_MAP)
xg['away_team'] = xg['away_team'].replace(NAME_MAP)

# Verifieer mapping
xg_teams   = set(xg['home_team']) | set(xg['away_team'])
train_teams = set(train['home_team']) | set(train['away_team'])
still_missing = xg_teams - train_teams
print('Nog niet gemapte teams:', still_missing if still_missing else 'Geen — alles matched!')

In [ ]:
# === MERGE xG AAN TRAININGSDATA ===
train = train.merge(
    xg[['date', 'home_team', 'away_team', 'home_xg', 'away_xg']],
    on=['date', 'home_team', 'away_team'],
    how='left'
)

matched = train['home_xg'].notna().sum()
print(f'Gekoppeld: {matched}/{len(train)} ({matched/len(train)*100:.1f}%)')
print(f'Niet gekoppeld: {len(train) - matched} (seizoenen zonder xG data)')

In [ ]:
# === ROLLING XG STATS BEREKENEN ===
# Sorteer op datum zodat rolling correct werkt
train = train.sort_values('timestamp').reset_index(drop=True)

# Initialiseer nieuwe kolommen
new_cols = [
    # Overall (thuis + uit samen)
    'home_xg_for_alltime_overall',  'home_xg_for_last10_overall',
    'home_xg_against_alltime_overall', 'home_xg_against_last10_overall',
    'away_xg_for_alltime_overall',  'away_xg_for_last10_overall',
    'away_xg_against_alltime_overall', 'away_xg_against_last10_overall',
    # Alleen thuiswedstrijden
    'home_xg_for_alltime_home',     'home_xg_for_last10_home',
    'home_xg_against_alltime_home', 'home_xg_against_last10_home',
    # Alleen uitwedstrijden
    'away_xg_for_alltime_away',     'away_xg_for_last10_away',
    'away_xg_against_alltime_away', 'away_xg_against_last10_away',
]
for col in new_cols:
    train[col] = np.nan

print('Kolommen aangemaakt, bezig met berekenen...')

In [ ]:
# === HOOFD BEREKENING ===
# Per team houden we een geschiedenis bij van gespeelde wedstrijden
# Dit spiegelt exact hoe de andere rolling stats in de dataset zijn opgebouwd:
# de waarde op rij k is het gemiddelde van ALLE wedstrijden VOOR wedstrijd k

from collections import defaultdict

# Geschiedenis per team: lijst van (xg_for, xg_against, venue)
# venue = 'home' of 'away'
history = defaultdict(list)  # team -> [(xg_scored, xg_conceded, venue), ...]

def rolling_mean(records, venue_filter=None, last_n=None):
    """Bereken gemiddelde xG over historische wedstrijden.
    venue_filter: None=overall, 'home', 'away'
    last_n: None=alltime, 10=last10
    """
    if venue_filter:
        records = [r for r in records if r[2] == venue_filter]
    if last_n:
        records = records[-last_n:]
    if not records:
        return np.nan
    return np.mean([r[0] for r in records]), np.mean([r[1] for r in records])

for idx, row in train.iterrows():
    home = row['home_team']
    away = row['away_team']
    h_xg = row['home_xg']
    a_xg = row['away_xg']

    # Sla de waarden op VOOR deze wedstrijd (pre-match)
    for team, xg_for, xg_against, venue, prefix in [
        (home, h_xg, a_xg, 'home', 'home'),
        (away, a_xg, h_xg, 'away', 'away'),
    ]:
        hist = history[team]

        # Overall alltime
        res = rolling_mean(hist)
        if res is not np.nan:
            train.at[idx, f'{prefix}_xg_for_alltime_overall']     = res[0]
            train.at[idx, f'{prefix}_xg_against_alltime_overall'] = res[1]

        # Overall last10
        res = rolling_mean(hist, last_n=10)
        if res is not np.nan:
            train.at[idx, f'{prefix}_xg_for_last10_overall']     = res[0]
            train.at[idx, f'{prefix}_xg_against_last10_overall'] = res[1]

        # Venue-specifiek alltime
        res = rolling_mean(hist, venue_filter=venue)
        if res is not np.nan:
            train.at[idx, f'{prefix}_xg_for_alltime_{venue}']     = res[0]
            train.at[idx, f'{prefix}_xg_against_alltime_{venue}'] = res[1]

        # Venue-specifiek last10
        res = rolling_mean(hist, venue_filter=venue, last_n=10)
        if res is not np.nan:
            train.at[idx, f'{prefix}_xg_for_last10_{venue}']     = res[0]
            train.at[idx, f'{prefix}_xg_against_last10_{venue}'] = res[1]

    # Update geschiedenis NA het opslaan (pre-match logica)
    if pd.notna(h_xg) and pd.notna(a_xg):
        history[home].append((h_xg, a_xg, 'home'))
        history[away].append((a_xg, h_xg, 'away'))

    if idx % 500 == 0:
        print(f'  {idx}/{len(train)} rijen verwerkt...')

print('Klaar!')

In [ ]:
# === VERIFICATIE ===
print('Nieuwe kolommen:')
print(train[new_cols].describe().round(3))

print('\nVoorbeeld rij (Liverpool thuis):') 
ex = train[train['home_team'] == 'Liverpool'][new_cols[:4]].dropna().head(5)
print(ex)

In [4]:
# === OPSLAAN ===
import pandas as pd
import os
os.chdir('C:/Users/semwi/FPL-Core-Insights/data/XGboost data')

# === OPSLAAN ===
train.to_csv('training_data_with_xg.csv', index=False)
print(f'Opgeslagen als training_data_with_xg.csv')
print(f'Totaal kolommen: {len(train.columns)}')
# Verwijder de regel met new_cols — die is al opgeslagen

Opgeslagen als training_data_with_xg.csv
Totaal kolommen: 507
